## Testing different prompts and hyperparameters

In [1]:
import subprocess
import json
import textwrap
from pathlib import Path
from datetime import datetime, timezone

In [2]:
import os
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

In [3]:
# Models to test
BASE_MODEL = "tinyllama"

# Temperatures to test
TEMPERATURES = [0.1, 0.3, 0.5]

# Prompt variants
PROMPTS = {
    "minimal": """Write a professional psychological report summary based on the input.""",

    "structured": """You are a licensed psychologist.
Write a professional psychoeducational report summary.
Use cautious, neutral clinical language.
Do not invent facts.
Base conclusions only on the provided data.""",

    "structured_fewshot": """You are a licensed psychologist.

Example:
Input:
WISC-V: VCI=85, WMI=78
Output:
The child demonstrated relative weaknesses in working memory, which may impact academic functioning.

Now write a report summary based on the following input."""
}

# Context lengths (simulate context window by truncation)
CONTEXT_LENGTHS = [512, 1024, 2048]

# Output directory
RESULTS_DIR = Path("ollama_experiments")
RESULTS_DIR.mkdir(exist_ok=True)


In [4]:
TEST_INPUT = """
Intake notes:
The child is a 9-year-old referred due to concerns about attention and academic performance.
Teachers report difficulty sustaining attention and completing tasks.

Test results:
WISC-V:
VCI: 92
WMI: 78
PSI: 85
FSIQ: 88
"""


In [5]:
def create_modelfile(model_name: str, system_prompt: str, temperature: float) -> Path:
    modelfile = f"""
FROM {BASE_MODEL}

SYSTEM \"\"\"
{system_prompt}
\"\"\"

PARAMETER temperature {temperature}
"""
    path = RESULTS_DIR / f"Modelfile_{model_name}"
    path.write_text(textwrap.dedent(modelfile).strip())
    return path


In [6]:
def build_model(model_name: str, modelfile_path):
    subprocess.run(
        ["ollama", "create", model_name, "-f", str(modelfile_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )


In [7]:
def run_ollama(model_name: str, prompt: str) -> str:
    process = subprocess.run(
        ["ollama", "run", model_name],
        input=prompt,
        text=True,
        capture_output=True
    )
    return process.stdout.strip()


In [ ]:
results = []

for prompt_name, system_prompt in PROMPTS.items():
    for temp in TEMPERATURES:
        model_name = f"tinyllama_{prompt_name}_t{temp}".replace(".", "_")

        modelfile_path = create_modelfile(model_name, system_prompt, temp)
        build_model(model_name, modelfile_path)

        for ctx_len in CONTEXT_LENGTHS:
            truncated_input = TEST_INPUT[:ctx_len]

            full_prompt = f"""
{system_prompt}

INPUT:
{truncated_input}

TASK:
Write a cognitive and attention-related summary.
"""

            output = run_ollama(model_name, full_prompt)

            record = {
                "model": model_name,
                "prompt_variant": prompt_name,
                "temperature": temp,
                "context_length": ctx_len,
                "output": output,
                "timestamp": datetime.now(timezone.utc).isoformat()
            }

            results.append(record)

            # Save individual output (explicit UTF-8 for Windows safety)
            out_file = RESULTS_DIR / f"{model_name}_ctx{ctx_len}.txt"
            out_file.write_text(output, encoding="utf-8")

print(f"Completed {len(results)} runs.")

Exception in thread Thread-22 (_readerthread):
Traceback (most recent call last):
  File "C:\Python314\Lib\threading.py", line 1081, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "C:\Python314\Lib\threading.py", line 1023, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Python314\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 440: character maps to <undefined>


In [ ]:
with open(RESULTS_DIR / "all_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved.")


In [ ]:
def show_outputs(prompt_variant, temperature):
    filtered = [
        r for r in results
        if r["prompt_variant"] == prompt_variant and r["temperature"] == temperature
    ]
    for r in filtered:
        print("=" * 80)
        print(f"Context length: {r['context_length']}")
        print(r["output"])
